# Projeto 2 — Data Lake Meteorológico de Fortaleza

## 1. Introdução

Este projeto tem como objetivo desenvolver uma solução de Engenharia de Dados em Cloud para coleta, armazenamento, processamento e análise de informações meteorológicas da cidade de Fortaleza.

A coleta dos dados é realizada por meio da API Open-Meteo, utilizando Python para obter informações como temperatura, umidade relativa do ar e velocidade do vento.

A solução utiliza serviços da AWS para implementar um fluxo automatizado de ingestão e armazenamento de dados. O Amazon S3 é utilizado como Data Lake, organizado em camadas de processamento, enquanto AWS Lambda e Amazon EventBridge permitem automatizar as coletas meteorológicas.

Os dados são catalogados pelo AWS Glue e disponibilizados para consultas analíticas utilizando Amazon Athena e SQL.

## 2. Objetivos

- Consumir dados meteorológicos por meio de uma API.
- Realizar a coleta de informações utilizando Python.
- Manipular e organizar dados com Pandas.
- Armazenar os dados no Amazon S3.
- Implementar automação utilizando AWS Lambda e EventBridge.
- Organizar os dados nas camadas Bronze, Silver e Gold.
- Catalogar os dados com AWS Glue.
- Executar consultas analíticas com Amazon Athena.
- Desenvolver visualizações dos indicadores meteorológicos.

## 3. Tecnologias utilizadas

- Python.
- Pandas.
- API Open-Meteo.
- Google Colab.
- AWS Lambda.
- Amazon EventBridge.
- Amazon S3.
- AWS Glue.
- Amazon Athena.
- Power BI e Lovable.

## 4. Arquitetura do projeto

API Open-Meteo → Python/AWS Lambda → S3 Bronze → S3 Silver → S3 Gold → AWS Glue → Amazon Athena → Dashboard.

## 5. Resultado esperado

Desenvolver uma solução capaz de coletar dados meteorológicos automaticamente, armazená-los em um Data Lake na AWS e disponibilizar indicadores para análise e visualização, permitindo acompanhar informações climáticas de Fortaleza.

In [ ]:

import requests
import pandas as pd


In [ ]:

# 1. Endereço da API
url = "https://api.open-meteo.com/v1/forecast"

# 2. Parâmetros da consulta
params = {
    "latitude": -3.7172,
    "longitude": -38.5433,
    "current": "temperature_2m,relative_humidity_2m,wind_speed_10m",
    "timezone": "America/Fortaleza"
}

# 3. Enviar a requisição
response = requests.get(url, params=params)

# 4. Verificar o resultado
print("Status da requisição:", response.status_code)


Status da requisição: 200


In [ ]:
response.json()

{'latitude': -3.7609842,
 'longitude': -38.529663,
 'generationtime_ms': 0.07176399230957031,
 'utc_offset_seconds': -10800,
 'timezone': 'America/Fortaleza',
 'timezone_abbreviation': 'GMT-3',
 'elevation': 14.0,
 'current_units': {'time': 'iso8601',
  'interval': 'seconds',
  'temperature_2m': '°C',
  'relative_humidity_2m': '%',
  'wind_speed_10m': 'km/h'},
 'current': {'time': '2026-11-09T22:30',
  'interval': 900,
  'temperature_2m': 27.9,
  'relative_humidity_2m': 73,
  'wind_speed_10m': 13.8}}

In [ ]:

# 1. Converter a resposta JSON em dicionário Python
dados = response.json()

# 2. Visualizar os dados recebidos
print(dados)


{'latitude': -3.7609842, 'longitude': -38.529663, 'generationtime_ms': 0.07176399230957031, 'utc_offset_seconds': -10800, 'timezone': 'America/Fortaleza', 'timezone_abbreviation': 'GMT-3', 'elevation': 14.0, 'current_units': {'time': 'iso8601', 'interval': 'seconds', 'temperature_2m': '°C', 'relative_humidity_2m': '%', 'wind_speed_10m': 'km/h'}, 'current': {'time': '2026-11-09T22:30', 'interval': 900, 'temperature_2m': 27.9, 'relative_humidity_2m': 73, 'wind_speed_10m': 13.8}}


In [ ]:

# Acessar os dados meteorológicos atuais
clima_atual = dados["current"]

# Exibir somente temperatura, umidade e vento
print(clima_atual)


{'time': '2026-11-09T22:30', 'interval': 900, 'temperature_2m': 27.9, 'relative_humidity_2m': 73, 'wind_speed_10m': 13.8}


In [ ]:

# Criar uma tabela com os dados meteorológicos
df_clima = pd.DataFrame([{
    "cidade": "Fortaleza",
    "temperatura": clima_atual["temperature_2m"],
    "umidade": clima_atual["relative_humidity_2m"],
    "vento": clima_atual["wind_speed_10m"]
}])

# Exibir a tabela
display(df_clima)


,cidade,temperatura,umidade,vento
0,Fortaleza,27.9,73,13.8


In [ ]:

# Importar biblioteca para trabalhar com datas e horários
from datetime import datetime

# Adicionar a data e hora da coleta
df_clima["data_coleta"] = datetime.now()

# Visualizar a tabela atualizada
display(df_clima)


,cidade,temperatura,umidade,vento,data_coleta
0,Fortaleza,27.9,73,13.8,2026-11-10 01:41:23.153771


In [ ]:

# Salvar os dados meteorológicos em CSV
df_clima.to_csv(
    "clima_fortaleza.csv",
    index=False
)

print("Arquivo CSV criado com sucesso!")


Arquivo CSV criado com sucesso!


In [ ]:

!pip install pyarrow


In [ ]:

# Salvar os dados meteorológicos em Parquet
df_clima.to_parquet(
    "clima_fortaleza.parquet",
    index=False
)

print("Arquivo Parquet criado com sucesso!")


Arquivo Parquet criado com sucesso!


In [ ]:
import os

print(os.listdir())

['.config', 'clima_fortaleza.parquet', 'clima_fortaleza.csv', 'sample_data']


In [ ]:
df_parquet = pd.read_parquet("clima_fortaleza.parquet")

display(df_parquet)

,cidade,temperatura,umidade,vento,data_coleta
0,Fortaleza,27.9,73,13.8,2026-11-10 01:41:23.153771


In [ ]:

# Criar o histórico inicial utilizando nossa tabela atual
df_historico = df_clima.copy()

# Visualizar o histórico
display(df_historico)


,cidade,temperatura,umidade,vento,data_coleta
0,Fortaleza,27.9,73,13.8,2026-11-10 01:41:23.153771


In [ ]:

# 1. Fazer uma nova requisição à API
response = requests.get(url, params=params)

# 2. Verificar se a requisição funcionou
response.raise_for_status()

# 3. Converter a resposta para dicionário
dados_novos = response.json()

# 4. Extrair os dados meteorológicos
clima_novo = dados_novos["current"]

# 5. Criar uma tabela com a nova coleta
nova_coleta = pd.DataFrame([{
    "cidade": "Fortaleza",
    "temperatura": clima_novo["temperature_2m"],
    "umidade": clima_novo["relative_humidity_2m"],
    "vento": clima_novo["wind_speed_10m"],
    "data_coleta": clima_novo["time"]
}])

# 6. Adicionar a coleta ao histórico
df_historico = pd.concat(
    [df_historico, nova_coleta],
    ignore_index=True
)

# 7. Visualizar o histórico
display(df_historico)


,cidade,temperatura,umidade,vento,data_coleta
0,Fortaleza,27.9,73,13.8,2026-11-10 01:41:23.153771
1,Fortaleza,27.9,72,13.8,2026-11-09T22:45


In [ ]:
# Converter a coluna de datas para o tipo datetime padronizado
df_historico["data_coleta"] = pd.to_datetime(df_historico["data_coleta"])

# Salvar o histórico acumulado
df_historico.to_parquet(
    "historico_clima_fortaleza.parquet",
    index=False
)

print("Histórico meteorológico atualizado com sucesso!")

Histórico meteorológico atualizado com sucesso!


In [ ]:
print("Total de coletas:", len(df_historico))

Total de coletas: 2


In [ ]:
# Visualizar todas as coletas
display(df_historico)

# Verificar quantidade de registros
print("Quantidade de registros:", len(df_historico))

# Verificar os tipos de dados
df_historico.info()

,cidade,temperatura,umidade,vento,data_coleta
0,Fortaleza,27.9,73,13.8,2026-11-10 01:41:23.153771
1,Fortaleza,27.9,72,13.8,2026-11-09 22:45:00.000000


Quantidade de registros: 2
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   cidade       2 non-null      object        
 1   temperatura  2 non-null      float64       
 2   umidade      2 non-null      int64         
 3   vento        2 non-null      float64       
 4   data_coleta  2 non-null      datetime64[ns]
dtypes: datetime64[ns](1), float64(2), int64(1), object(1)
memory usage: 212.0+ bytes


In [ ]:
# Salvar o histórico meteorológico em Parquet
df_historico.to_parquet(
    "bronze_clima_fortaleza.parquet",
    index=False
)

print("Arquivo Bronze criado com sucesso!")

Arquivo Bronze criado com sucesso!


In [ ]:

from google.colab import files

files.download("bronze_clima_fortaleza.parquet")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>